## Génération des dimensions à partir des fichiers accidents_fr

### Dimension localisation

In [ ]:
import pandas as pd

CHUNK_SIZE = 100_000
ID_PAYS = 1  # France

CATR_MAP = {
    1: "Autoroute",
    2: "Route nationale",
    3: "Route départementale",
    4: "Voie communale",
    5: "Hors réseau public",
    6: "Parc de stationnement",
    7: "Routes de métropole urbaine",
    9: "Autre",
}

# ── Generic chunked processor (same function as in dim_uk.py)
def process_chunks(csv_path, usecols, transform_fn, dedup_cols, label):
    print(f"\n[{label}] {csv_path}")
    seen, rows, total = set(), [], 0

    for i, chunk in enumerate(pd.read_csv(
            csv_path, encoding="utf-8-sig",
            usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False)):
        total += len(chunk)
        chunk = transform_fn(chunk)
        for row in chunk[dedup_cols].drop_duplicates().itertuples(index=False):
            key = tuple(row)
            if key not in seen:
                seen.add(key)
                rows.append(row)
        print(f"  chunk {i+1}: {total:,} read | {len(rows):,} unique")
    dim = pd.DataFrame(rows, columns=dedup_cols).reset_index(drop=True)
    print(f"  => {len(dim):,} rows")
    return dim

# ── 1. DIM_LOCALISATION ────────────────────────────────────────────────────────

def transform_localisation_fr(c):
    c["type_route"]     = pd.to_numeric(c["catr"], errors="coerce").map(CATR_MAP).fillna("Unknown")
    c["vitesse_limite"] = pd.to_numeric(c["vma"],  errors="coerce").replace(-1, pd.NA)
    c["commune"]        = c["com"].astype(str)
    c["departement"]    = c["dep"].astype(str)
    c["region"]         = "N/A"
    c["id_pays"]        = ID_PAYS
    c["latitude"]       = pd.to_numeric(c["lat"],  errors="coerce").round(4)
    c["longitude"]      = pd.to_numeric(c["long"], errors="coerce").round(4)
    return c

dim_loc = process_chunks(
    "../../data/combined/jointure.csv",
    ["lat", "long", "catr", "vma", "com", "dep"],
    transform_localisation_fr,
    ["id_pays", "commune", "departement", "region", "type_route", "vitesse_limite", "latitude", "longitude"],
    "DIM_LOCALISATION"
)
dim_loc.insert(0, "id_lieu", range(1, len(dim_loc) + 1))
dim_loc[["id_lieu", "id_pays", "commune", "departement", "region",
         "type_route", "vitesse_limite", "latitude", "longitude"]].to_csv("../../data/processed/dim_localisation_fr.csv", index=False)
print("  Saved -> /data/processed/dim_localisation_fr.csv")



[DIM_LOCALISATION] ../../data/combined/jointure.csv
  chunk 1: 100,000 read | 8,394 unique
  chunk 2: 200,000 read | 19,641 unique
  chunk 3: 300,000 read | 28,862 unique
  chunk 4: 400,000 read | 37,137 unique
  chunk 5: 500,000 read | 46,446 unique
  chunk 6: 600,000 read | 55,436 unique
  chunk 7: 700,000 read | 65,091 unique
  chunk 8: 800,000 read | 72,487 unique
  chunk 9: 900,000 read | 81,428 unique
  chunk 10: 1,000,000 read | 88,790 unique
  chunk 11: 1,100,000 read | 103,234 unique
  chunk 12: 1,200,000 read | 114,827 unique
  chunk 13: 1,300,000 read | 122,093 unique
  chunk 14: 1,400,000 read | 135,983 unique
  chunk 15: 1,500,000 read | 147,891 unique
  chunk 16: 1,600,000 read | 157,950 unique
  chunk 17: 1,700,000 read | 177,719 unique
  chunk 18: 1,800,000 read | 184,864 unique
  chunk 19: 1,900,000 read | 206,259 unique
  chunk 20: 2,000,000 read | 227,290 unique
  chunk 21: 2,100,000 read | 241,923 unique
  chunk 22: 2,200,000 read | 252,299 unique
  chunk 23: 2,300

### Dimension usager

In [ ]:
SEX_MAP = {
    1: "Masculin",
    2: "Féminin",
}

SEVERITY_MAP = {
    1: "Indemne",
    2: "Tué",
    3: "Blessé hospitalisé",
    4: "Blessé léger",
}

CASUALTY_CLASS_MAP = {
    1: "Conducteur",
    2: "Passager",
    3: "Piéton",
}

PLACE_MAP = {
    1: "Avant gauche", 2: "Avant droit", 3: "Arrière gauche",
    4: "Arrière droit", 5: "Arrière centre", 6: "Avant centre",
    7: "Nez à nez",    8: "Arrière",       9: "Autre",
}

def transform_usager_fr(c):
    c["age"]        = (c["an"] + 2000).where(c["an"] < 100, c["an"]) - pd.to_numeric(c["an_nais"], errors="coerce")
    c["age"]        = c["age"].where(c["age"] >= 0, pd.NA)
    c["sexe"]       = pd.to_numeric(c["sexe"], errors="coerce").map(SEX_MAP).fillna("Unknown")
    c["gravite"]    = pd.to_numeric(c["grav"],  errors="coerce").map(SEVERITY_MAP).fillna("Unknown")
    c["cat_usager"] = pd.to_numeric(c["catu"],  errors="coerce").map(CASUALTY_CLASS_MAP).fillna("Unknown")
    c["place_vehicule"] = pd.to_numeric(c["place"], errors="coerce").map(PLACE_MAP).fillna("Unknown")
    c["id_pays"]    = ID_PAYS
    return c

dim_usg = process_chunks(
    "../../data/combined/jointure.csv",
    ["an", "an_nais", "sexe", "catu", "grav", "place"],
    transform_usager_fr,
    ["id_pays", "age", "sexe", "place_vehicule", "gravite", "cat_usager"],
    "DIM_USAGER"
)
dim_usg.insert(0, "id_usager", range(1, len(dim_usg) + 1))
dim_usg[["id_usager", "id_pays", "age", "sexe",
         "place_vehicule", "gravite", "cat_usager"]].to_csv("../../data/processed/dim_usager_fr.csv", index=False)
print("  Saved -> ../../data/processed/dim_usager_fr.csv")


[DIM_USAGER] ../../data/combined/jointure.csv
  chunk 1: 100,000 read | 3,324 unique
  chunk 2: 200,000 read | 4,410 unique
  chunk 3: 300,000 read | 5,015 unique
  chunk 4: 400,000 read | 5,434 unique
  chunk 5: 500,000 read | 5,779 unique
  chunk 6: 600,000 read | 6,103 unique
  chunk 7: 700,000 read | 6,294 unique
  chunk 8: 800,000 read | 6,493 unique
  chunk 9: 900,000 read | 6,686 unique
  chunk 10: 1,000,000 read | 6,846 unique
  chunk 11: 1,100,000 read | 6,996 unique
  chunk 12: 1,200,000 read | 7,125 unique
  chunk 13: 1,300,000 read | 7,204 unique
  chunk 14: 1,400,000 read | 7,329 unique
  chunk 15: 1,500,000 read | 7,433 unique
  chunk 16: 1,600,000 read | 7,528 unique
  chunk 17: 1,700,000 read | 7,644 unique
  chunk 18: 1,800,000 read | 7,698 unique
  chunk 19: 1,900,000 read | 8,016 unique
  chunk 20: 2,000,000 read | 8,096 unique
  chunk 21: 2,100,000 read | 8,181 unique
  chunk 22: 2,200,000 read | 8,303 unique
  chunk 23: 2,300,000 read | 8,410 unique
  chunk 24: 2,

### Dimension véhicule

In [ ]:
VEHICLE_TYPE_MAP = {
    1: "Bicyclette", 2: "Cyclomoteur <50cm3", 3: "Voiturette",
    4: "Référence inutilisée", 5: "Référence inutilisée", 6: "Référence inutilisée",
    7: "VL seul", 8: "Référence inutilisée", 9: "Référence inutilisée",
    10: "VU seul 1,5T<=PTAC<=3,5T", 11: "Référence inutilisée", 12: "Référence inutilisée",
    13: "PL seul 3,5T<PTAC<=7,5T", 14: "PL seul >7,5T", 15: "PL >3,5T + remorque",
    16: "Tracteur routier seul", 17: "Tracteur routier + semi-remorque",
    18: "Référence inutilisée", 19: "Référence inutilisée", 20: "Engin spécial",
    21: "Tracteur agricole", 30: "Scooter <50cm3", 31: "Motocyclette >50cm3 <=125cm3",
    32: "Scooter >50cm3 <=125cm3", 33: "Motocyclette >125cm3", 34: "Scooter >125cm3",
    35: "Quad léger <=50cm3", 36: "Quad lourd >50cm3", 37: "Autobus",
    38: "Autocar", 39: "Train", 40: "Tramway", 41: "3RM <=50cm3",
    42: "3RM >50cm3 <=125cm3", 43: "3RM >125cm3", 50: "EDP à moteur",
    60: "EDP sans moteur", 80: "VAE", 99: "Autre véhicule",
}

MANOEUVRE_MAP = {
    1: "Sans changement de direction", 2: "Même sens, même file",
    3: "Entre 2 files", 4: "En marche arrière",
    5: "A contresens", 6: "En franchissant le terre-plein central",
    7: "Dans le couloir bus, dans le même sens",
    8: "Dans le couloir bus, dans le sens inverse",
    9: "En s'insérant", 10: "En faisant demi-tour",
    11: "Changeant de file à gauche", 12: "Changeant de file à droite",
    13: "Déporté à gauche", 14: "Déporté à droite",
    15: "Tournant à gauche", 16: "Tournant à droite",
    17: "Dépassant à gauche", 18: "Dépassant à droite",
    19: "Traversant la chaussée", 20: "Manœuvre de stationnement",
    21: "Manœuvre d'évitement", 22: "Ouverture de porte",
    23: "Arrêté (hors stationnement)", 24: "En stationnement (avec occupants)",
    25: "Circulant sur trottoir", 26: "Autres manœuvres",
}

PROPULSION_MAP = {
    0: "Inconnue", 1: "Hydrocarbures", 2: "Hybride électrique",
    3: "Electrique", 4: "Hydrogène", 5: "Humaine",
    6: "Autre",
}

def transform_vehicule_fr(c):
    c["type_vehicule"] = pd.to_numeric(c["catv"],  errors="coerce").map(VEHICLE_TYPE_MAP).fillna("Unknown")
    c["manoeuvre"]     = pd.to_numeric(c["manv"],  errors="coerce").map(MANOEUVRE_MAP).fillna("Unknown")
    c["motorisation"]  = pd.to_numeric(c["motor"], errors="coerce").map(PROPULSION_MAP).fillna("Unknown")
    c["nb_occupants"]  = pd.to_numeric(c["occutc"], errors="coerce").replace(-1, pd.NA)
    c["id_pays"]       = ID_PAYS
    return c

dim_veh = process_chunks(
    "../../data/combined/jointure.csv",
    ["catv", "manv", "motor", "occutc"],
    transform_vehicule_fr,
    ["id_pays", "type_vehicule", "manoeuvre", "nb_occupants", "motorisation"],
    "DIM_VEHICULE"
)
dim_veh.insert(0, "id_vehicule", range(1, len(dim_veh) + 1))
dim_veh[["id_vehicule", "id_pays", "type_vehicule", "manoeuvre",
         "nb_occupants", "motorisation"]].to_csv("../../data/processed/dim_vehicule_fr.csv", index=False)
print("  Saved -> ../../data/processed/dim_vehicule_fr.csv")


[DIM_VEHICULE] ../../data/combined/jointure.csv
  chunk 1: 100,000 read | 397 unique
  chunk 2: 200,000 read | 481 unique
  chunk 3: 300,000 read | 525 unique
  chunk 4: 400,000 read | 692 unique
  chunk 5: 500,000 read | 819 unique
  chunk 6: 600,000 read | 901 unique
  chunk 7: 700,000 read | 967 unique
  chunk 8: 800,000 read | 1,060 unique
  chunk 9: 900,000 read | 1,123 unique
  chunk 10: 1,000,000 read | 1,177 unique
  chunk 11: 1,100,000 read | 1,209 unique
  chunk 12: 1,200,000 read | 1,260 unique
  chunk 13: 1,300,000 read | 1,305 unique
  chunk 14: 1,400,000 read | 1,349 unique
  chunk 15: 1,500,000 read | 1,383 unique
  chunk 16: 1,600,000 read | 1,411 unique
  chunk 17: 1,700,000 read | 1,442 unique
  chunk 18: 1,800,000 read | 1,467 unique
  chunk 19: 1,900,000 read | 1,502 unique
  chunk 20: 2,000,000 read | 1,536 unique
  chunk 21: 2,100,000 read | 2,254 unique
  chunk 22: 2,200,000 read | 3,390 unique
  chunk 23: 2,300,000 read | 4,531 unique
  chunk 24: 2,400,000 read